# grok-thermostat: pre-registered closed-loop WD campaign
**Branch:** `feature/grok-thermostat` · protocol: README gate + amendment A9 · details: CAMPAIGN.md
**One click:** Runtime -> Run all. Results go to Google Drive (`MyDrive/moonshot_results/grok-thermostat`); re-running after a disconnect resumes from the last finished stage. The GPU is unassigned at the end or on any error.

In [ ]:
# Step 1: Credit guard (unassign GPU on any error or at the end)
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt or completion: unassigning GPU...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)
print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 2: GPU check
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU (L4 or A100)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | torch {torch.__version__}")

In [ ]:
# Step 3: Clone (or update) the repository and install
import os, subprocess
repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/grok-thermostat"
target_dir = "/content/grok-thermostat"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "checkout", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
%cd /content/grok-thermostat
!pip install -q -r requirements.txt

In [ ]:
# Step 4: Tests (fast, CPU)
%cd /content/grok-thermostat
r = subprocess.run(["python", "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed"

In [ ]:
# Step 5: Persist results/ on Google Drive (survives disconnects; campaign.py resumes from finished stages)
import shutil
from google.colab import drive
drive.mount("/content/drive")
drive_dir = "/content/drive/MyDrive/moonshot_results/grok-thermostat"
os.makedirs(drive_dir, exist_ok=True)
res = "/content/grok-thermostat/results"
if os.path.isdir(res) and not os.path.islink(res):
    shutil.copytree(res, drive_dir, dirs_exist_ok=True)
    shutil.rmtree(res)
if not os.path.islink(res):
    os.symlink(drive_dir, res)
print("results ->", os.path.realpath(res), "| existing:", sorted(os.listdir(res)))

In [ ]:
# Step 6: Full campaign (calibration -> dev grids -> H1/controller selection -> eval -> verdict)
%cd /content/grok-thermostat
!python -u campaign.py 2>&1 | tee -a results/campaign.log
assert os.path.exists("results/verdict.json"), "campaign did not finish; re-run all to resume"

In [ ]:
# Step 7: Results
print(open("/content/grok-thermostat/results/RESULTS.md").read())

In [ ]:
# Step 8: Unassign GPU (zero credit leakage)
auto_teardown()